<a href="https://colab.research.google.com/github/knkillname/exploraciones/blob/main/cuadernos/enoe2026-t1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # La Encuesta Nacional de Ocupación y Empleo (ENOE)
 La **Encuesta Nacional de Ocupación y Empleo (ENOE)** es la principal fuente de
 información sobre el mercado laboral mexicano. Ofrece datos mensuales y trimestrales
 sobre la fuerza de trabajo, ocupación, informalidad, subocupación y desocupación.

 ### Antecedentes y Evolución
 - **Inicio:** Enero de 2005, consolidando la ENEU y la ENE.
 - **Adaptaciones:** Durante la contingencia sanitaria (abril-junio 2020) se implementó la ETOE (Telefónica). Entre 2020 y 2022 se utilizó la edición ENOEN y, a partir de 2023, se regresó al formato ENOE estándar.
 - **Importancia:** En 2011 fue declarada **Información de Interés Nacional**, siendo oficial y de uso obligatorio para la toma de decisiones y diseño de políticas públicas.

 ### Objetivos y Cobertura
 **Objetivo General:**
 Obtener información estadística sobre la fuerza de trabajo y las características
 ocupacionales de la población a nivel nacional, estatal y por ciudades.
 **Población Objetivo:**
 Personas residentes habituales de las viviendas. Aunque se captan datos desde los 12
 años, los indicadores oficiales se generan para la población de **15 años y más**
 (edad legal mínima para trabajar en México).
 **Cobertura Geográfica:**
 - Nacional.
 - 32 Entidades federativas.
 - 39 Ciudades autorrepresentadas.

 ### Diseño Metodológico
 - **Periodicidad:** Trimestral.
 - **Esquema de Muestreo:** Probabilístico, bietápico, estratificado y por conglomerados.
 - **Panel Rotatorio:** La muestra se divide en 5 paneles; cada vivienda es visitada durante 5 trimestres. Cada trimestre se renueva el 20% de la muestra.
 - **Tamaño de Muestra:** Aproximadamente 150,000 viviendas trimestrales.
 - **Instrumentos:** Cuestionario sociodemográfico y Cuestionario de ocupación y empleo (básico y ampliado).
 Esta encuesta sigue recomendaciones internacionales de la **OIT** y la **OCDE**, permitiendo comparabilidad internacional y un análisis profundo de la informalidad y presión en el mercado laboral.

In [ ]:
import csv
import sqlite3
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns



In [ ]:
def download(url: str, path: Path, chunk_size: int | None = 8192) -> None:
    """Descarga un archivo desde una URL y lo guarda en la ruta especificada."""
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                f.write(chunk)
    else:
        raise RuntimeError(f"Error {response.status_code}")


def extract_zip(zip_path: Path, extract_dir: Path) -> None:
    """Extrae un archivo ZIP en el directorio especificado."""
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)



In [ ]:
url = (
    "https://www.inegi.org.mx/contenidos/programas/enoe/15ymas/"
    "datosabiertos/2026/conjunto_de_datos_enoe_2026_1t_csv.zip"
)
file_path = Path(url.rsplit("/", maxsplit=1)[-1])
data_dir = Path("enoe_data")

if file_path.exists():
    print(f"El archivo '{file_path}' ya existe; omitiendo descarga.")
else:
    print("Descargando datos de la encuesta...")
    try:
        download(url, file_path)
    except Exception as e:
        print(f"Error al descargar los datos: {e}")
    else:
        print("Descarga exitosa")

print("Extrayendo archivos...")
data_dir.mkdir(parents=True, exist_ok=True)
extract_zip(file_path, data_dir)
print("Extracción completada")


 Catálogo de las tablas contenidas:

In [ ]:
pd.read_csv(data_dir / "0_indice_tablas_enoe_2026_1t.csv", skiprows=2)


 ## Base de datos relacional en SQLite3

 Vamos a construir una base de datos relacional a partir de los archivos
 CSV descargados. Para lograrlo necesitamos tres ingredientes que el
 INEGI incluye en cada entrega de la ENOE:
 Base de datos relacional en SQLite3
 **Diccionario de datos.** Un CSV que describe cada columna: su
 nemónico (nombre corto), su tipo (`C` = código/texto, `N` = número
 entero) y el catálogo al que pertenece, si aplica.

 **Catálogo.** Una tabla de referencia que traduce códigos a
 descripciones legibles. Por ejemplo, el catálogo `cve_ent` convierte
 `"01"` en `"Aguascalientes"` y `cd_a` convierte `"14"` en `"México"`.
 Sin catálogos, los datos serían sólo números sin significado.

 **Jerarquía muestral.** La ENOE sigue un diseño en tres niveles:
 vivienda → hogar → persona. Esta jerarquía determina las llaves
 primarias compuestas y las llaves foráneas entre tablas.

 ### Diccionarios de datos

 Cada una de las 5 tablas de la ENOE viene acompañada de un diccionario
 de datos. Lo leemos con `csv.reader` para extraer tres datos por
 columna: nemónico, tipo SQL y catálogo asociado (si tiene).

In [ ]:
# Llaves primarias del diseño muestral ENOE (fijas, no cambian entre ediciones)
PK_VIV = ["cd_a", "cve_ent", "con", "upm", "d_sem", "n_pro_viv", "v_sel", "n_ent"]
PK_HOG = PK_VIV + ["n_hog"]
PK_PER = PK_HOG + ["n_ren"]

TABLA_DIRS: dict[str, str] = {
    "viv": "conjunto_de_datos_viv_enoe_2026_1t",
    "hog": "conjunto_de_datos_hog_enoe_2026_1t",
    "sdem": "conjunto_de_datos_sdem_enoe_2026_1t",
    "coe1": "conjunto_de_datos_coe1_enoe_2026_1t",
    "coe2": "conjunto_de_datos_coe2_enoe_2026_1t",
}
PKS: dict[str, list[str]] = {
    "viv": PK_VIV,
    "hog": PK_HOG,
    "sdem": PK_PER,
    "coe1": PK_PER,
    "coe2": PK_PER,
}
PADRE: dict[str, str | None] = {
    "viv": None,
    "hog": "viv",
    "sdem": "hog",
    "coe1": "hog",
    "coe2": "hog",
}



In [ ]:
def leer_diccionario(nombre: str) -> list[dict[str, str]]:
    """Lee el diccionario de datos de una tabla y devuelve columnas con tipo SQL."""
    ruta = (
        data_dir
        / TABLA_DIRS[nombre]
        / "diccionario_de_datos"
        / f"diccionario_datos_{nombre}_enoe_2026_1t.csv"
    )
    with ruta.open(encoding="utf-8", newline="") as f:
        filas = list(csv.reader(f))
    idx = {h.strip(): i for i, h in enumerate(filas[0])}
    return [
        {
            "nem": r[idx["NEMÓNICO"]].strip(),
            "tipo": "INTEGER" if r[idx["TIPO"]].strip() == "N" else "TEXT",
            "cat": r[idx["CATÁLOGO"]].strip(),
        }
        for r in filas[1:]
        if r[idx["NEMÓNICO"]].strip()
    ]


columnas = {n: leer_diccionario(n) for n in TABLA_DIRS}
for nombre, cols in columnas.items():
    n_int = sum(1 for c in cols if c["tipo"] == "INTEGER")
    print(f"  {nombre}: {len(cols)} columnas ({n_int} INTEGER)")


 ### Creación de las tablas

 Con las columnas ya inferidas, generamos el `CREATE TABLE` para cada
 tabla. La llave primaria compuesta refleja el diseño muestral: una
 vivienda se identifica por 8 columnas (cd_a, cve_ent, con, upm…); un
 hogar añade `n_hog`; una persona añade `n_ren`. La llave foránea
 asegura que cada hogar pertenezca a una vivienda real y cada persona
 a un hogar real (integridad referencial).

In [ ]:
def crear_tablas(
    conn: sqlite3.Connection,
    columnas: dict[str, list[dict[str, str]]],
    pks: dict[str, list[str]],
    padre: dict[str, str | None],
) -> None:
    """Crea las 5 tablas con PK compuestas y FK jerárquicas."""
    for nombre, cols in columnas.items():
        nems_tabla = {c["nem"] for c in cols}
        partes = [f"    {c['nem']} {c['tipo']}" for c in cols]
        pk = [c for c in pks[nombre] if c in nems_tabla]  # PK compuesta
        partes.append(f"    PRIMARY KEY ({', '.join(pk)})")
        if (p := padre[nombre]) is not None:  # FK a la tabla padre
            fk_cols = [c for c in pks[p] if c in nems_tabla]
            partes.append(
                f"    FOREIGN KEY ({', '.join(fk_cols)})"
                f" REFERENCES {p}({', '.join(fk_cols)})"
            )
        conn.execute(f"CREATE TABLE {nombre} (\n" + ",\n".join(partes) + "\n)")
        print(f"  ✓ {nombre} creada")


DB_PATH = Path("enoe.db")
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(str(DB_PATH))
conn.execute("PRAGMA foreign_keys = ON")
conn.execute("PRAGMA journal_mode = WAL")
crear_tablas(conn, columnas, PKS, PADRE)


 ### Carga de datos

 Los archivos de datos usan codificación `latin-1` (a diferencia de los
 diccionarios y catálogos, que van en `utf-8`). Para no saturar la
 memoria con tablas de cientos de miles de filas, leemos por lotes de
 50 000 filas (`chunksize`) y filtramos solo las columnas que aparecen
 en el diccionario de datos.

In [ ]:
def cargar_datos(
    conn: sqlite3.Connection,
    data_dir: Path,
    tabla_dirs: dict[str, str],
    columnas: dict[str, list[dict[str, str]]],
    chunksize: int = 50_000,
) -> None:
    """Carga los CSV de la ENOE en las tablas ya creadas, por lotes."""
    for nombre, subdir in tabla_dirs.items():
        archivo = (
            data_dir
            / subdir
            / "conjunto_de_datos"
            / f"conjunto_de_datos_{nombre}_enoe_2026_1t.csv"
        )
        nems = [c["nem"] for c in columnas[nombre]]
        total = 0
        for chunk in pd.read_csv(
            archivo,
            encoding="latin-1",
            dtype=str,
            keep_default_na=False,
            na_values=[],
            chunksize=chunksize,
        ):
            chunk = chunk[[c for c in nems if c in chunk.columns]]
            chunk.to_sql(nombre, conn, if_exists="append", index=False)
            total += len(chunk)
        print(f"  {nombre}: {total:,} filas, {len(nems)} columnas")


cargar_datos(conn, data_dir, TABLA_DIRS, columnas)


 ### Catálogos

 Un catálogo es una tabla de consulta que responde «¿qué significa este
 código?». Por ejemplo, en la columna `sex` de la tabla SDEM aparece
 `"1"` o `"2"`; el catálogo `sex` nos dice que `"1"` = hombre y
 `"2"` = mujer. Sin estos catálogos, los datos serían ilegibles.

 Los catálogos vienen en archivos `cve,descrip` (a veces con columnas
 extra como `CVEGEO`). Detectamos la columna que contiene la clave
 comparando la cabecera con el nombre del catálogo. La descripción
 siempre es la última columna. No usamos `PRIMARY KEY` en catálogos
 porque algunas claves como `cve_mun` (municipio) se repiten entre
 entidades federativas.

In [ ]:
def buscar_archivo_catalogo(
    data_dir: Path, tabla_dirs: dict[str, str], cat: str
) -> Path | None:
    """Busca el CSV de un catálogo en los 5 subdirectorios de datos."""
    return next(
        (
            data_dir / sd / "catalogos" / f"{cat}.csv"
            for sd in tabla_dirs.values()
            if (data_dir / sd / "catalogos" / f"{cat}.csv").exists()
        ),
        None,
    )



In [ ]:
def cargar_catalogos(
    conn: sqlite3.Connection,
    data_dir: Path,
    tabla_dirs: dict[str, str],
    columnas: dict[str, list[dict[str, str]]],
) -> int:
    """Carga los catálogos en streaming y devuelve el número de tablas creadas."""
    cats_unicos = {c["cat"] for cols in columnas.values() for c in cols if c["cat"]}
    cargados = 0
    for cat in sorted(cats_unicos):
        archivo = buscar_archivo_catalogo(data_dir, tabla_dirs, cat)
        if archivo is None:
            continue
        with archivo.open(encoding="utf-8", newline="") as f:
            filas = list(csv.reader(f))
        cabecera = [h.strip().lower() for h in filas[0]]
        idx_key = next(  # "cve" o el nombre del catálogo
            (i for i, h in enumerate(cabecera) if h in ("cve", cat.lower())), 0
        )
        idx_desc = len(cabecera) - 1  # descripción = última columna
        datos = [
            (r[idx_key].strip(), r[idx_desc].strip())
            for r in filas[1:]
            if len(r) > max(idx_key, idx_desc) and r[idx_key].strip()
        ]
        sql_create = (
            f"CREATE TABLE IF NOT EXISTS catalogo_{cat}"
            f" (cve TEXT, descripcion TEXT)"
        )
        sql_insert = f"INSERT INTO catalogo_{cat} (cve, descripcion) VALUES (?, ?)"
        conn.execute(sql_create)
        conn.executemany(sql_insert, datos)
        cargados += 1
    conn.commit()  # executemany no hace autocommit en sqlite3
    return cargados



 La base de datos ya está lista para consultas SQL. Podemos hacer
 JOINs entre las 5 tablas principales y decodificar cualquier columna
 usando su catálogo. Por ejemplo, para contar personas por entidad
 federativa haríamos un JOIN entre SDEM y el catálogo `cve_ent`.

In [ ]:
catalogos_cargados = cargar_catalogos(conn, data_dir, TABLA_DIRS, columnas)
conn.close()
tam_mb = DB_PATH.stat().st_size / (1024 * 1024)
print(f"  {catalogos_cargados} catálogos cargados")
print(f"\n✓ Base de datos lista en {DB_PATH.resolve()} ({tam_mb:.1f} MB)")

# Verificación: abrimos en modo lectura y consultamos el total de personas
conn_ro = sqlite3.connect(str(DB_PATH))
n_personas = conn_ro.execute("SELECT COUNT(*) FROM sdem").fetchone()[0]
print(f"  Total de personas en SDEM: {n_personas:,}")
conn_ro.close()


 ## Análisis exploratorio

 Con la base de datos relacional lista, pasamos al análisis exploratorio.
 Todas las consultas usan SQL con JOINs a los catálogos para obtener
 etiquetas legibles en español. Los conteos se ponderan con `fac_tri`
 (factor de expansión trimestral) para producir estimaciones poblacionales,
 no solo frecuencias de muestra.

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
sns.set_theme(style="whitegrid", palette="muted")


def consultar(sql: str, conn: sqlite3.Connection = conn) -> pd.DataFrame:
    """Ejecuta una consulta SQL y devuelve un DataFrame."""
    return pd.read_sql_query(sql, conn)



 ### Estadísticos descriptivos

 Antes de graficar conviene mirar los números. Para las variables
 numéricas clave —edad, escolaridad, horas trabajadas e ingreso— se
 calculan los estadísticos ponderados por `fac_tri` (media, desviación
 estándar, cuartiles) para la población de 15 años y más.

In [ ]:
def estadisticos_descriptivos(conn: sqlite3.Connection) -> pd.DataFrame:
    """Estadísticos descriptivos ponderados para variables numéricas clave."""
    consultas: dict[str, str] = {
        "Edad (años)": "SELECT CAST(s.eda AS REAL) FROM sdem s"
        " WHERE CAST(s.eda AS INTEGER) >= 15",
        "Años de escolaridad": "SELECT CAST(s.anios_esc AS REAL) FROM sdem s"
        " WHERE CAST(s.eda AS INTEGER) >= 15 AND s.anios_esc IS NOT NULL",
        "Horas trabajadas": "SELECT CAST(s.hrsocup AS REAL) FROM sdem s WHERE s.clase1 = '1'",
        "Ingreso mensual": "SELECT CAST(s.ingocup AS REAL) FROM sdem s"
        " WHERE s.clase1 = '1' AND s.ingocup IS NOT NULL",
    }
    filas: list[dict[str, object]] = []
    for nombre, sql in consultas.items():
        serie = pd.read_sql_query(sql, conn).iloc[:, 0]
        filas.append(
            {
                "Variable": nombre,
                "n": len(serie),
                "Media": round(serie.mean(), 1),
                "Std": round(serie.std(), 1),
                "Mín": round(serie.min(), 1),
                "Q1": round(serie.quantile(0.25), 1),
                "Mediana": round(serie.median(), 1),
                "Q3": round(serie.quantile(0.75), 1),
                "Máx": round(serie.max(), 1),
            }
        )
    return pd.DataFrame(filas)


print(estadisticos_descriptivos(conn).to_string(index=False))


 ### Demografía

 #### Pirámide de población por condición de empleo

 Como corresponde a una encuesta de empleo, cada barra de la pirámide
 se descompone en pilas (stacks) según la condición de actividad:
 ocupado, desocupado, inactivo y no aplica (menores de 15 años o sin
 dato). Así se aprecia de un vistazo cuánta gente trabaja en cada
 grupo de edad y sexo.

In [ ]:
def graficar_piramide(df: pd.DataFrame) -> None:
    """Pirámide con barras apiladas por condición de actividad."""
    colores = {
        "Ocupado": "#2E86AB",
        "Desocupado": "#C44E52",
        "Inactivo": "#9E9E9E",
        "No aplica": "#E0E0E0",
    }
    orden = ["Ocupado", "Desocupado", "Inactivo", "No aplica"]
    fig, ax = plt.subplots(figsize=(10, 6.5))
    for sexo, signo in [("Hombre", -1), ("Mujer", 1)]:
        sub = df[df["sexo"] == sexo].pivot_table(
            index="grupo_edad",
            columns="condicion",
            values="poblacion",
            aggfunc="sum",
            fill_value=0,
        )
        left = 0.0
        for cond in orden:
            if cond not in sub.columns:
                continue
            valores = sub[cond] * signo
            ax.barh(
                sub.index,
                valores,
                left=left * signo,
                color=colores[cond],
                height=0.78,
                label=cond if sexo == "Hombre" else "",
            )
            left += sub[cond]
    ax.set_xlabel("Población (millones)")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{abs(v):.1f}"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=8, ncol=2, loc="lower right")



In [ ]:
df_piramide = consultar("""
    SELECT
        CASE WHEN CAST(s.eda AS INTEGER) < 5  THEN '00-04'
             WHEN CAST(s.eda AS INTEGER) < 10 THEN '05-09'
             WHEN CAST(s.eda AS INTEGER) < 15 THEN '10-14'
             WHEN CAST(s.eda AS INTEGER) < 20 THEN '15-19'
             WHEN CAST(s.eda AS INTEGER) < 25 THEN '20-24'
             WHEN CAST(s.eda AS INTEGER) < 30 THEN '25-29'
             WHEN CAST(s.eda AS INTEGER) < 35 THEN '30-34'
             WHEN CAST(s.eda AS INTEGER) < 40 THEN '35-39'
             WHEN CAST(s.eda AS INTEGER) < 45 THEN '40-44'
             WHEN CAST(s.eda AS INTEGER) < 50 THEN '45-49'
             WHEN CAST(s.eda AS INTEGER) < 55 THEN '50-54'
             WHEN CAST(s.eda AS INTEGER) < 60 THEN '55-59'
             WHEN CAST(s.eda AS INTEGER) < 65 THEN '60-64'
             WHEN CAST(s.eda AS INTEGER) < 70 THEN '65-69'
             WHEN CAST(s.eda AS INTEGER) < 75 THEN '70-74'
             WHEN CAST(s.eda AS INTEGER) < 80 THEN '75-79'
             WHEN CAST(s.eda AS INTEGER) < 85 THEN '80-84'
             ELSE '85+'
        END AS grupo_edad,
        cs.descripcion AS sexo,
        CASE WHEN s.clase1 = '1' THEN 'Ocupado'
             WHEN s.clase1 = '2' THEN 'Desocupado'
             WHEN s.clase1 IN ('3', '4') THEN 'Inactivo'
             ELSE 'No aplica'
        END AS condicion,
        SUM(s.fac_tri) / 1000000.0 AS poblacion
    FROM sdem s JOIN catalogo_sex cs ON s.sex = cs.cve
    GROUP BY grupo_edad, sexo, condicion
""")
graficar_piramide(df_piramide)


 #### Escolaridad por sexo

 El nivel de instrucción (`niv_edu`) se decodifica con su catálogo
 para obtener categorías legibles. Las barras agrupadas por sexo
 permiten comparar sin los picos y el solapamiento de un KDE sobre
 datos discretos.

In [ ]:
def graficar_escolaridad(df: pd.DataFrame) -> None:
    """Barras agrupadas de nivel educativo por sexo."""
    fig, ax = plt.subplots(figsize=(8, 5))
    azul, naranja = "#4A6FA5", "#C44E52"
    sns.barplot(
        data=df,
        x="poblacion",
        y="nivel",
        hue="sexo",
        palette={"Hombre": azul, "Mujer": naranja},
        ax=ax,
    )
    ax.set_xlabel("Población estimada (millones)")
    ax.set_ylabel("")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=9)


df_esc = consultar("""
    SELECT cn.descripcion AS nivel, cs.descripcion AS sexo,
           SUM(s.fac_tri) / 1000000.0 AS poblacion
    FROM sdem s
    JOIN catalogo_niv_ins cn ON s.niv_ins = cn.cve
    JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.eda >= 15
    GROUP BY s.niv_ins, s.sex
    ORDER BY cn.cve
""")
graficar_escolaridad(df_esc)


 ### Mercado laboral

 #### Condición de actividad

 La ENOE clasifica a la población de 15+ en PEA (ocupada y desocupada)
 y PNEA (disponible y no disponible). El catálogo `clase1` decodifica
 estas cuatro categorías. Una barra horizontal por categoría, sin
 saturación cromática, con el valor anotado directamente.

In [ ]:
def graficar_clase1(df: pd.DataFrame) -> None:
    """Barras apiladas horizontales de condición de actividad por sexo."""
    piv = df.pivot_table(
        index="condicion",
        columns="sexo",
        values="poblacion",
        aggfunc="sum",
        fill_value=0,
    )
    orden = piv.sum(axis=1).sort_values(ascending=True).index
    piv = piv.loc[orden]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    azul, naranja = "#4A6FA5", "#C44E52"
    left = 0.0
    for sexo, color in [("Hombre", azul), ("Mujer", naranja)]:
        ax.barh(piv.index, piv[sexo], left=left, color=color, height=0.6, label=sexo)
        left += piv[sexo].values
    ax.set_xlabel("Población estimada (millones)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.tick_params(left=False)
    ax.legend(frameon=False, fontsize=9)


df_clase1 = consultar("""
    SELECT cc.descripcion AS condicion, cs.descripcion AS sexo,
           SUM(s.fac_tri) / 1000000.0 AS poblacion
    FROM sdem s
    JOIN catalogo_clase1 cc ON s.clase1 = cc.cve
    JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.eda >= 15
    GROUP BY s.clase1, s.sex
""")
graficar_clase1(df_clase1)


 #### Distribución del ingreso mensual

 El ingreso de la población ocupada tiene una distribución muy
 asimétrica (cola larga a la derecha). Una curva KDE con escala
 logarítmica permite comparar las formas entre hombres y mujeres
 mejor que un boxplot.

In [ ]:
def graficar_ingreso(df: pd.DataFrame) -> None:
    """Curvas de densidad del ingreso por sexo, escala logarítmica."""
    df = df[df["ingocup"] > 0]
    fig, ax = plt.subplots(figsize=(8, 4))
    azul, naranja = "#4A6FA5", "#C44E52"
    for sexo, color in [("Hombre", azul), ("Mujer", naranja)]:
        sub = df[df["sexo"] == sexo]
        sns.kdeplot(
            data=sub,
            x="ingocup",
            color=color,
            fill=True,
            alpha=0.2,
            linewidth=2,
            ax=ax,
            label=sexo,
            log_scale=True,
            bw_adjust=0.8,
        )
    ax.set_xlabel("Ingreso mensual (escala log)")
    ax.set_ylabel("Densidad")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=10)


df_ingreso = consultar("""
    SELECT CAST(s.ingocup AS REAL) AS ingocup, cs.descripcion AS sexo
    FROM sdem s JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.clase1 = '1'
""")
graficar_ingreso(df_ingreso)


 #### Edad e ingreso

 La relación entre edad e ingreso muestra el ciclo de vida laboral:
 el ingreso crece en la juventud, alcanza una meseta en la edad
 productiva y declina en la tercera edad. La gráfica se corta a los
 80 años porque más allá la muestra es demasiado pequeña para que
 la mediana y los cuartiles sean fiables.

In [ ]:
def graficar_edad_ingreso(df: pd.DataFrame) -> None:
    """Mediana e intervalo intercuartil del ingreso por edad (15−80 años)."""
    df = df[(df["ingocup"] > 0) & (df["eda"] >= 15) & (df["eda"] <= 80)]
    agg = df.groupby("eda")["ingocup"].agg(
        q1=lambda x: x.quantile(0.25),
        mediana="median",
        q3=lambda x: x.quantile(0.75),
    )
    edades = agg.index.astype(int)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.fill_between(edades, agg["q1"], agg["q3"], color="#4A6FA5", alpha=0.15,
                    label="Rango intercuartil (Q1–Q3)")
    ax.plot(edades, agg["mediana"], color="#4A6FA5", lw=2, label="Mediana")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1000:,.0f}k"))
    ax.set_xlabel("Edad (años)")
    ax.set_ylabel("Ingreso mensual (miles de pesos)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3, lw=0.5)
    ax.set_ylim(0, None)
    ax.legend(frameon=False, fontsize=9)


df_ed_ing = consultar("""
    SELECT CAST(s.eda AS INTEGER) AS eda,
           CAST(s.ingocup AS REAL) AS ingocup,
           cs.descripcion AS sexo
    FROM sdem s JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.clase1 = '1'
""")
graficar_edad_ingreso(df_ed_ing)


 #### Posición en la ocupación

 ¿Asalariados, cuenta propia o negocio familiar? El catálogo `pos_ocu`
 desglosa las categorías. Barras agrupadas por sexo, sin saturación de
 color: solo dos tonos distinguibles.

In [ ]:
def graficar_pos_ocu(df: pd.DataFrame) -> None:
    """Barras agrupadas de posición en la ocupación por sexo."""
    fig, ax = plt.subplots(figsize=(8, 4))
    azul, naranja = "#4A6FA5", "#C44E52"
    sns.barplot(
        data=df,
        x="poblacion",
        y="posicion",
        hue="sexo",
        palette={"Hombre": azul, "Mujer": naranja},
        ax=ax,
    )
    ax.set_xlabel("Población estimada (millones)")
    ax.set_ylabel("")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=9)


df_pos = consultar("""
    SELECT cp.descripcion AS posicion, cs.descripcion AS sexo,
           SUM(s.fac_tri) / 1000000.0 AS poblacion
    FROM sdem s
    JOIN catalogo_pos_ocu cp ON s.pos_ocu = cp.cve
    JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.clase1 = '1'
    GROUP BY s.pos_ocu, s.sex
    ORDER BY cp.cve
""")
graficar_pos_ocu(df_pos)


 ### Geografía

 #### Población de 15 años y más por entidad

 La ENOE estima para las 32 entidades. Una barra horizontal por estado,
 ordenada de mayor a menor, con un solo color. Sin grid vertical
 innecesaria ni bordes decorativos.

In [ ]:
def graficar_entidades(df: pd.DataFrame) -> None:
    """Barras horizontales de población por entidad, una barra por estado."""
    orden = df.sort_values("poblacion")
    fig, ax = plt.subplots(figsize=(7, 8))
    color = "#4A6FA5"
    ax.barh(orden["entidad"], orden["poblacion"], color=color, height=0.7)
    ax.set_xlabel("Población estimada (millones)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.tick_params(axis="y", labelsize=8)


df_ent = consultar("""
    SELECT ce.descripcion AS entidad,
           SUM(s.fac_tri) / 1000000.0 AS poblacion
    FROM sdem s JOIN catalogo_cve_ent ce ON s.cve_ent = ce.cve
    WHERE s.eda >= 15
    GROUP BY s.cve_ent
""")
graficar_entidades(df_ent)


 #### Tasa de desocupación por entidad

 Cociente entre PEA desocupada y PEA total. Las barras se ordenan de
 menor a mayor tasa; el valor porcentual se anota al final de cada barra.

In [ ]:
def graficar_desocupacion(df: pd.DataFrame) -> None:
    """Barras horizontales de tasa de desocupación con etiquetas directas."""
    orden = df.sort_values("tasa")
    fig, ax = plt.subplots(figsize=(7, 8))
    color = "#C44E52"
    ax.barh(orden["entidad"], orden["tasa"], color=color, height=0.7)
    for bar in ax.containers[0]:
        ax.text(
            bar.get_width() + 0.15,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%",
            va="center",
            fontsize=7.5,
            color="#555",
        )
    ax.set_xlabel("Tasa de desocupación (%)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.tick_params(axis="y", labelsize=8)


df_des = consultar("""
    SELECT ce.descripcion AS entidad,
           100.0 * SUM(CASE WHEN s.clase1 = '2' THEN s.fac_tri ELSE 0 END)
                 / SUM(CASE WHEN s.clase1 IN ('1', '2') THEN s.fac_tri ELSE 0 END)
           AS tasa
    FROM sdem s JOIN catalogo_cve_ent ce ON s.cve_ent = ce.cve
    WHERE s.eda >= 15
    GROUP BY s.cve_ent
    HAVING tasa > 0
""")
graficar_desocupacion(df_des)


 ## Historias del mercado laboral mexicano

 Más allá de los números, los datos de la ENOE cuentan historias sobre
 cómo trabajan los mexicanos. En esta sección respondemos once preguntas
 usando la base de datos relacional que construimos. Cada gráfico sigue
 los principios de Tufte: solo la tinta necesaria para que los datos
 hablen por sí mismos.

 ### Acto I — El mapa del empleo

 #### El mapa del empleo: sectores y formalidad

 El catálogo `rama` clasifica a los ocupados en 7 sectores económicos.
 Cada barra se divide en empleo **formal** e **informal** según
 `emp_ppal`, la clasificación oficial de la ENOE. La anotación muestra
 el ingreso mensual mediano del sector completo.

 El cuestionario de la ENOE permite además caracterizar el *tipo* de
 trabajo desde muchos otros ángulos: quién es el cliente (`p3c1-p3c4`),
 si hay contrato (`p3j`), si es temporal (`p3k1`), el tamaño de la
 empresa (`p3q`), el turno (`p5`), o la pertenencia a un sindicato
 (`p3i`). Aquí elegimos la formalidad como eje por ser el indicador
 más sintético de calidad del empleo.

In [ ]:
def graficar_sectores(df: pd.DataFrame) -> None:
    """Barras apiladas de población ocupada por sector y formalidad."""
    piv = df.pivot_table(
        index="sector",
        columns="tipo",
        values="poblacion",
        aggfunc="sum",
        fill_value=0,
    )
    ingreso = df.groupby("sector")["ingreso_mediano"].first()
    piv = piv.loc[piv.sum(axis=1).sort_values(ascending=True).index]
    fig, ax = plt.subplots(figsize=(8, 4))
    left = 0.0
    for cond, color in [("Formal", "#2E86AB"), ("Informal", "#E0E0E0")]:
        ax.barh(
            piv.index, piv.get(cond, 0), left=left, color=color, height=0.6, label=cond
        )
        left += piv.get(cond, 0)
    for i, sector in enumerate(piv.index):
        ax.text(
            piv.sum(axis=1).iloc[i] + 0.3,
            i,
            f"${ingreso[sector]:,.0f}",
            va="center",
            fontsize=8,
            color="#333",
        )
    ax.set_xlabel("Población ocupada (millones)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1, 1))
    ax.tick_params(left=False)



In [ ]:
df_rama = consultar("""
    SELECT cr.descripcion AS sector,
           CASE s.emp_ppal WHEN '1' THEN 'Formal' ELSE 'Informal' END AS tipo,
           SUM(s.fac_tri) / 1000000.0 AS poblacion,
           AVG(CAST(s.ingocup AS REAL)) AS ingreso_mediano
    FROM sdem s JOIN catalogo_rama cr ON s.rama = cr.cve
    WHERE s.clase1 = '1'
    GROUP BY s.rama, tipo
""")
graficar_sectores(df_rama)


 #### ¿Desde dónde se trabaja?

 `p4h` de COE1 clasifica el lugar habitual de trabajo. En vez de un
 boxplot —que resulta plano por la enorme categoría «Local fijo»—,
 mostramos la mediana con su rango intercuartil y el tamaño de cada
 grupo. El ingreso más alto en «Vehículo» probablemente refleja
 transportistas, repartidores y taxistas que reportan ingreso bruto
 (sin descontar gasolina ni mantenimiento).

In [ ]:
def graficar_lugar_trabajo(df: pd.DataFrame) -> None:
    """Punto y rango intercuartil de ingreso por lugar de trabajo."""
    stats = (
        df.groupby("lugar")["ingocup"]
        .agg(
            mediana="median",
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
            n="count",
        )
        .sort_values("mediana")
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    for i, (lugar, row) in enumerate(stats.iterrows()):
        ax.plot([row["q1"], row["q3"]], [i, i], color="#4A6FA5", lw=2, alpha=0.6)
        ax.scatter(row["mediana"], i, color="#4A6FA5", s=40, zorder=3)
        ax.text(
            row["q3"] * 1.05,
            i,
            f"n={row['n']:,}",
            va="center",
            fontsize=8,
            color="#666",
        )
    ax.set_yticks(range(len(stats)))
    ax.set_yticklabels(stats.index)
    ax.set_xscale("log")
    ax.set_xlabel("Ingreso mensual (escala log)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.tick_params(left=False)



In [ ]:
df_lugar = consultar("""
    SELECT CASE c.p4h WHEN '1' THEN 'En casa' WHEN '2' THEN 'Local fijo'
                WHEN '3' THEN 'Distintos lugares' WHEN '4' THEN 'Puesto improvisado'
                WHEN '5' THEN 'Vehículo' ELSE 'Otro/NS'
           END AS lugar,
           CAST(s.ingocup AS REAL) AS ingocup
    FROM sdem s
    JOIN coe1 c ON s.cd_a = c.cd_a AND s.cve_ent = c.cve_ent
        AND s.con = c.con AND s.upm = c.upm AND s.d_sem = c.d_sem
        AND s.n_pro_viv = c.n_pro_viv AND s.v_sel = c.v_sel
        AND s.n_ent = c.n_ent AND s.n_hog = c.n_hog AND s.n_ren = c.n_ren
    WHERE s.clase1 = '1' AND CAST(s.ingocup AS REAL) > 0
""")
graficar_lugar_trabajo(df_lugar)


 ### Acto II — Educación y dinero

 #### ¿Estudiar paga?

 Cada punto en el eje horizontal es un año de escolaridad (0 a 24).
 La línea muestra la **mediana** del ingreso mensual; la banda sombreada
 abarca del primer al tercer cuartil (el 50% central de los ingresos).
 Las líneas punteadas marcan los hitos del sistema educativo mexicano:
 primaria (6 años), secundaria (9), preparatoria (12), licenciatura
 —incluyendo la Normal Superior— (16) y posgrado (20).

In [ ]:
def graficar_educacion_ingreso(df: pd.DataFrame) -> None:
    """Línea de ingreso mediano por año de escolaridad, con banda Q1–Q3."""
    agrupado = df.groupby("anios_esc")["ingocup"].quantile([0.25, 0.5, 0.75]).unstack()
    agrupado.columns = ["q1", "mediana", "q3"]
    agrupado = agrupado.reset_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.fill_between(
        agrupado["anios_esc"],
        agrupado["q1"],
        agrupado["q3"],
        color="#4A6FA5",
        alpha=0.15,
    )
    ax.plot(agrupado["anios_esc"], agrupado["mediana"], color="#4A6FA5", lw=2.5)
    hitos = {
        6: "Primaria",
        9: "Secundaria",
        12: "Preparatoria",
        16: "Licenciatura / Normal superior",
        20: "Posgrado",
    }
    for anios, etiqueta in hitos.items():
        if anios <= agrupado["anios_esc"].max():
            ax.axvline(anios, color="#C44E52", lw=1, ls="--", alpha=0.5)
            ax.text(
                anios + 0.2,
                agrupado["mediana"].max() * 0.92,
                etiqueta,
                rotation=90,
                va="top",
                fontsize=7.5,
                color="#C44E52",
                alpha=0.8,
            )
    ax.set_xlabel("Años de escolaridad aprobados")
    ax.set_ylabel("Ingreso mensual (mediana y rango intercuartil)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3, lw=0.5)



In [ ]:
df_edu_ing = consultar("""
    SELECT CAST(s.anios_esc AS INTEGER) AS anios_esc,
           CAST(s.ingocup AS REAL) AS ingocup
    FROM sdem s
    WHERE s.clase1 = '1' AND s.ingocup IS NOT NULL
      AND CAST(s.anios_esc AS INTEGER) BETWEEN 0 AND 24
""")
graficar_educacion_ingreso(df_edu_ing)


 #### ¿La brecha de género se cierra con más educación?

 La gráfica de «¿Estudiar paga?» mostró que las mujeres estudian más
 años que los hombres. Sin embargo, en **todos** los niveles educativos
 ellas ganan menos. Esta doble gráfica revela una parte de la historia:
 la brecha de ingreso mensual (izquierda) persiste, pero las mujeres
 trabajan sistemáticamente **menos horas** que los hombres (derecha).
 ¿Se debe la brecha a que trabajan menos horas, o también ganan menos
 **por hora**? Lo exploramos en la siguiente celda.

In [ ]:
def graficar_brecha_educacion(df_ingreso: pd.DataFrame, df_horas: pd.DataFrame) -> None:
    """Panel doble: ingreso por sexo y nivel, y horas trabajadas."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    # Orden compartido: del catálogo niv_ins (primaria → superior)
    niveles = df_ingreso["nivel"].unique().tolist()
    # Panel izquierdo: ingreso mensual por sexo y nivel (barras agrupadas)
    sns.barplot(
        data=df_ingreso,
        x="ingreso",
        y="nivel",
        hue="sexo",
        order=niveles,
        palette={"Hombre": "#4A6FA5", "Mujer": "#C44E52"},
        ax=ax1,
    )
    ax1.set_xlabel("Ingreso mensual mediano")
    ax1.spines[["top", "right"]].set_visible(False)
    ax1.grid(axis="x", alpha=0.3, lw=0.5)
    ax1.tick_params(left=False)
    ax1.legend(frameon=False, fontsize=9)
    # Panel derecho: horas trabajadas por sexo y nivel
    for sexo, color, marker in [("Hombre", "#4A6FA5", "o"), ("Mujer", "#C44E52", "s")]:
        sub = df_horas[df_horas["sexo"] == sexo].set_index("nivel").loc[niveles]
        ax2.plot(
            sub["horas"],
            sub.index,
            color=color,
            marker=marker,
            lw=2,
            markersize=6,
            alpha=0.85,
            label=sexo,
        )
    ax2.set_xlabel("Horas trabajadas (mediana semanal)")
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.grid(axis="x", alpha=0.3, lw=0.5)
    ax2.tick_params(left=False)
    ax2.legend(frameon=False, fontsize=9)



In [ ]:
df_ingreso_edu = consultar("""
    SELECT cn.descripcion AS nivel, cs.descripcion AS sexo,
           AVG(CAST(s.ingocup AS REAL)) AS ingreso
    FROM sdem s
    JOIN catalogo_niv_ins cn ON s.niv_ins = cn.cve
    JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.clase1 = '1' AND s.ingocup IS NOT NULL
    GROUP BY s.niv_ins, s.sex ORDER BY cn.cve
""")

df_horas_edu = consultar("""
    SELECT cn.descripcion AS nivel, cs.descripcion AS sexo,
           AVG(CAST(s.hrsocup AS REAL)) AS horas
    FROM sdem s
    JOIN catalogo_niv_ins cn ON s.niv_ins = cn.cve
    JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.clase1 = '1' AND s.hrsocup IS NOT NULL
    GROUP BY s.niv_ins, s.sex ORDER BY cn.cve
""")
graficar_brecha_educacion(df_ingreso_edu, df_horas_edu)


 #### ¿Y por hora? El ingreso laboral por hora

 Al dividir el ingreso mensual entre las horas trabajadas obtenemos
 el salario por hora. Si la brecha de ingreso mensual se debiera
 únicamente a que las mujeres trabajan menos horas, las dos curvas
 deberían ser casi idénticas. El área sombreada muestra la brecha
 **por hora** que persiste entre hombres y mujeres.

In [ ]:
def graficar_brecha_hora(df: pd.DataFrame) -> None:
    """Salario por hora mediano y diferencia H−M por año de escolaridad."""
    agrupado = df.groupby(["anios_esc", "sexo"])["ing_x_hora"].median().unstack()
    h, m = agrupado.get("Hombre", 0), agrupado.get("Mujer", 0)
    diff = h - m
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(8, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]},
    )
    # Panel superior: curvas de salario por hora
    ax1.plot(agrupado.index, h, color="#4A6FA5", lw=2, label="Hombre")
    ax1.plot(agrupado.index, m, color="#C44E52", lw=2, label="Mujer")
    ax1.fill_between(agrupado.index, m, h, where=m < h, color="#C44E52", alpha=0.1)
    ax1.set_ylabel("Salario por hora (mediana)")
    ax1.spines[["top", "right"]].set_visible(False)
    ax1.grid(alpha=0.3, lw=0.5)
    ax1.legend(frameon=False, fontsize=10)
    # Panel inferior: diferencia H−M
    ax2.fill_between(agrupado.index, 0, diff, where=diff >= 0,
                     color="#4A6FA5", alpha=0.6, label="Hombre gana más")
    ax2.fill_between(agrupado.index, 0, diff, where=diff < 0,
                     color="#C44E52", alpha=0.6, label="Mujer gana más")
    ax2.axhline(0, color="#333", lw=0.8, alpha=0.5)
    ax2.set_xlabel("Años de escolaridad")
    ax2.set_ylabel("Diferencia H−M")
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.grid(alpha=0.3, lw=0.5)
    ax2.legend(frameon=False, fontsize=8)



In [ ]:
df_brecha_hora = consultar("""
    SELECT CAST(s.anios_esc AS INTEGER) AS anios_esc,
           cs.descripcion AS sexo,
           CAST(s.ingocup AS REAL) / NULLIF(CAST(s.hrsocup AS REAL), 0) AS ing_x_hora
    FROM sdem s JOIN catalogo_sex cs ON s.sex = cs.cve
    WHERE s.clase1 = '1' AND s.ingocup IS NOT NULL AND s.hrsocup IS NOT NULL
      AND CAST(s.anios_esc AS INTEGER) BETWEEN 0 AND 24
""")
graficar_brecha_hora(df_brecha_hora)


 #### ¿Dónde trabajan los más educados?

 Un mapa de calor que cruza nivel educativo con sector económico:
 ¿la industria atrae más ingenieros? ¿los servicios más profesionistas?

In [ ]:
def graficar_educacion_sector(df: pd.DataFrame) -> None:
    """Heatmap: porcentaje de cada nivel educativo dentro de cada sector."""
    piv = df.pivot_table(
        index="nivel",
        columns="sector",
        values="poblacion",
        aggfunc="sum",
        fill_value=0,
    )
    piv = piv.div(piv.sum(axis=0), axis=1) * 100
    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.heatmap(
        piv,
        annot=True,
        fmt=".0f",
        cmap="Blues",
        linewidths=0.5,
        cbar_kws={"label": "%"},
        ax=ax,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30, labelsize=8)


df_edu_sec = consultar("""
    SELECT cn.descripcion AS nivel, cr.descripcion AS sector,
           SUM(s.fac_tri) AS poblacion
    FROM sdem s
    JOIN catalogo_niv_ins cn ON s.niv_ins = cn.cve
    JOIN catalogo_rama cr ON s.rama = cr.cve
    WHERE s.clase1 = '1'
    GROUP BY s.niv_ins, s.rama
""")
graficar_educacion_sector(df_edu_sec)


 ### Acto III — Calidad del empleo

 #### Formales vs informales: ¿quién trabaja más horas?

 `emp_ppal` distingue empleo formal (1) del informal (2). Las curvas
 de densidad muestran la distribución de horas trabajadas en cada grupo.

In [ ]:
def graficar_horas_formalidad(df: pd.DataFrame) -> None:
    """KDE de horas trabajadas: formal vs informal."""
    fig, ax = plt.subplots(figsize=(8, 4))
    for cond, color, label in [
        ("Formal", "#2E86AB", "Formal"),
        ("Informal", "#C44E52", "Informal"),
    ]:
        sub = df[df["tipo"] == cond]
        sns.kdeplot(
            data=sub,
            x="hrsocup",
            color=color,
            fill=True,
            alpha=0.2,
            linewidth=2,
            ax=ax,
            label=label,
            bw_adjust=1.2,
        )
    ax.set_xlabel("Horas trabajadas en la semana")
    ax.set_ylabel("Densidad")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=10)


df_horas = consultar("""
    SELECT CASE s.emp_ppal WHEN '1' THEN 'Formal' ELSE 'Informal' END AS tipo,
           CAST(s.hrsocup AS REAL) AS hrsocup
    FROM sdem s WHERE s.clase1 = '1'
""")
graficar_horas_formalidad(df_horas)


 #### ¿Las empresas grandes dan más prestaciones?

 `p3q` de COE1 clasifica por número de empleados. Las barras muestran
 qué porcentaje de trabajadores en cada tamaño de empresa recibe
 aguinaldo, vacaciones pagadas y fondo de retiro (SAR/Afore).

In [ ]:
def graficar_prestaciones(df: pd.DataFrame) -> None:
    """Barras agrupadas: % con prestaciones por tamaño de empresa."""
    fig, ax = plt.subplots(figsize=(9, 4))
    df_melt = df.melt(id_vars="tamano", var_name="prestacion", value_name="porcentaje")
    sns.barplot(
        data=df_melt,
        x="tamano",
        y="porcentaje",
        hue="prestacion",
        palette={
            "Aguinaldo": "#2E86AB",
            "Vacaciones": "#4A6FA5",
            "SAR/Afore": "#C44E52",
        },
        ax=ax,
    )
    ax.set_xlabel("Tamaño de la empresa (nº empleados)")
    ax.set_ylabel("% de ocupados que reciben la prestación")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=9)



In [ ]:
df_prest = consultar("""
    SELECT CASE c.p3q WHEN '01' THEN '1 solo' WHEN '02' THEN '2-5'
                WHEN '03' THEN '6-10' WHEN '04' THEN '11-15'
                WHEN '05' THEN '16-50' WHEN '06' THEN '51-250'
                WHEN '07' THEN '251+' ELSE 'NS'
           END AS tamano,
           100.0 * SUM(CASE WHEN c.p3l1 = '1' THEN s.fac_tri ELSE 0 END)
               / SUM(s.fac_tri) AS "Aguinaldo",
           100.0 * SUM(CASE WHEN c.p3l2 = '2' THEN s.fac_tri ELSE 0 END)
               / SUM(s.fac_tri) AS "Vacaciones",
           100.0 * SUM(CASE WHEN c.p3m4 = '4' THEN s.fac_tri ELSE 0 END)
               / SUM(s.fac_tri) AS "SAR/Afore"
    FROM sdem s
    JOIN coe1 c ON s.cd_a = c.cd_a AND s.cve_ent = c.cve_ent
        AND s.con = c.con AND s.upm = c.upm AND s.d_sem = c.d_sem
        AND s.n_pro_viv = c.n_pro_viv AND s.v_sel = c.v_sel
        AND s.n_ent = c.n_ent AND s.n_hog = c.n_hog AND s.n_ren = c.n_ren
    WHERE s.clase1 = '1' AND c.p3q IS NOT NULL
    GROUP BY c.p3q
""")
graficar_prestaciones(df_prest)


 #### Subocupación: queriendo trabajar más

 `sub_o` identifica a quienes trabajan menos horas de las que
 quisieran. El porcentaje de subocupados varía notablemente entre
 entidades federativas.

In [ ]:
def graficar_subocupacion(df: pd.DataFrame) -> None:
    """Barras horizontales de % de subocupados por entidad."""
    orden = df.sort_values("pct_subocupados")
    fig, ax = plt.subplots(figsize=(7, 8))
    ax.barh(orden["entidad"], orden["pct_subocupados"], color="#C44E52", height=0.7)
    for bar in ax.containers[0]:
        ax.text(
            bar.get_width() + 0.15,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%",
            va="center",
            fontsize=7.5,
            color="#555",
        )
    ax.set_xlabel("% de ocupados que son subocupados")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.tick_params(axis="y", labelsize=8)


df_sub = consultar("""
    SELECT ce.descripcion AS entidad,
           100.0 * SUM(CASE WHEN s.sub_o = '1' THEN s.fac_tri ELSE 0 END)
               / SUM(s.fac_tri) AS pct_subocupados
    FROM sdem s JOIN catalogo_cve_ent ce ON s.cve_ent = ce.cve
    WHERE s.clase1 = '1'
    GROUP BY s.cve_ent
""")
graficar_subocupacion(df_sub)


 ### Acto IV — Familia, edad y trabajo

 #### El dilema familiar: ¿más hijos, más horas trabajadas?

 `hij5c` clasifica a las mujeres por número de hijos. Para las
 ocupadas, ¿la necesidad económica se traduce en una jornada más larga?

In [ ]:
def graficar_hijos_trabajo(df: pd.DataFrame) -> None:
    """Horas trabajadas medianas por número de hijos (mujeres ocupadas)."""
    orden = df.sort_values("hrsocup", ascending=True)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(orden["hijos"], orden["hrsocup"], color="#C44E52", height=0.6)
    for bar in ax.containers[0]:
        ax.text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f} h",
            va="center",
            fontsize=9,
            color="#333",
        )
    ax.set_xlabel("Horas trabajadas (mediana semanal)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3, lw=0.5)
    ax.tick_params(left=False)


df_hijos = consultar("""
    SELECT CASE s.hij5c WHEN '1' THEN 'Ninguno' WHEN '2' THEN '1 hijo'
                WHEN '3' THEN '2 hijos' WHEN '4' THEN '3 hijos'
                WHEN '5' THEN '4 o más' ELSE 'NS'
           END AS hijos,
           AVG(CAST(s.hrsocup AS REAL)) AS hrsocup
    FROM sdem s WHERE s.sex = '2' AND s.clase1 = '1'
    GROUP BY s.hij5c ORDER BY s.hij5c
""")
graficar_hijos_trabajo(df_hijos)


 #### ¿Quién hace qué? Edad y ocupación

 Cruzamos grupos de edad con la posición en la ocupación. ¿Los jóvenes
 son mayoritariamente asalariados? ¿A qué edad se emprende por cuenta
 propia?

In [ ]:
def graficar_edad_ocupacion(df: pd.DataFrame) -> None:
    """Barras apiladas: posición en la ocupación por grupo de edad."""
    piv = df.pivot_table(
        index="edad",
        columns="posicion",
        values="poblacion",
        aggfunc="sum",
        fill_value=0,
    )
    piv = piv.div(piv.sum(axis=1), axis=0) * 100
    colores = ["#2E86AB", "#4A6FA5", "#C44E52", "#9E9E9E", "#E0E0E0"]
    fig, ax = plt.subplots(figsize=(10, 5))
    left = 0.0
    for i, col in enumerate(piv.columns):
        ax.bar(
            piv.index,
            piv[col],
            bottom=left,
            color=colores[i % len(colores)],
            label=col,
            width=0.85,
        )
        left += piv[col].values
    ax.set_xlabel("Grupo de edad")
    ax.set_ylabel("% de ocupados")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=7.5, ncol=3, loc="upper right")



In [ ]:
df_edad_ocu = consultar("""
    SELECT CASE WHEN CAST(s.eda AS INTEGER) < 20 THEN '15-19'
                WHEN CAST(s.eda AS INTEGER) < 30 THEN '20-29'
                WHEN CAST(s.eda AS INTEGER) < 40 THEN '30-39'
                WHEN CAST(s.eda AS INTEGER) < 50 THEN '40-49'
                WHEN CAST(s.eda AS INTEGER) < 60 THEN '50-59'
                ELSE '60+'
           END AS edad,
           cp.descripcion AS posicion,
           SUM(s.fac_tri) AS poblacion
    FROM sdem s JOIN catalogo_pos_ocu cp ON s.pos_ocu = cp.cve
    WHERE s.clase1 = '1'
    GROUP BY edad, posicion
""")
graficar_edad_ocupacion(df_edad_ocu)


 ### Epílogo

 Once preguntas, once ventanas al mercado laboral mexicano en el primer
 trimestre de 2026. Los datos confirman algunas intuiciones —la
 educación paga, la formalidad trae prestaciones— y revelan matices
 —la brecha salarial persiste incluso con estudios, la subocupación
 tiene geografía—. Quedan muchas preguntas por responder; los datos
 están servidos en `enoe.db` para quien quiera continuar la exploración.